# Session 12: from `assert` lines to tests with `pytest`

In session 3 you checked your functions with `assert` lines at the bottom of
the file, and the slide said: "this is the seed of proper testing, which is
on the map in session 12 under `pytest`". Here it is.

Spoiler: you already know nearly all of it. A test is a function with
`assert` lines in it. `pytest` is the tool that finds those functions, runs
them all, and tells you which ones failed and why.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

## Where we left it in session 3

The four functions, and the checks you wrote under them.

In [ ]:
def total(numbers):
    """Return the sum of a list of numbers."""
    running = 0
    for number in numbers:
        running += number
    return running


def average(numbers):
    """Return the mean of a list of numbers."""
    return total(numbers) / len(numbers)


def highest(numbers):
    """Return the largest value in a list of numbers."""
    best = numbers[0]
    for number in numbers:
        if number > best:
            best = number
    return best


def lowest(numbers):
    """Return the smallest value in a list of numbers."""
    worst = numbers[0]
    for number in numbers:
        if number < worst:
            worst = number
    return worst


assert total([1, 2, 3]) == 6
assert average([2, 4]) == 3
assert highest([-5, -2, -9]) == -2          # the bug from session 2, pinned down
assert lowest([-5, -2, -9]) == -9
assert highest([7]) == 7
print("checks passed")

That worked well for one file. It has three problems once a project grows:

1. **It stops at the first failure.** If the second `assert` fails you
   learn nothing about the other three.
2. **It runs every time the file is imported.** The session 3 solution file
   also prints two summary lines at the bottom, so any other file that does
   `import statistics_functions` prints them too. A file that other code
   imports should only define things.
3. **Nobody runs it on purpose.** The checks run when you happen to run that
   file, not when you change something else that it depends on.

Tests fix all three by moving the checks into their own functions, in their
own file.

## A test is a function

Take each `assert` and give it a function of its own, named as a sentence
that says what should be true. The name must start with `test_`.

In [ ]:
def test_total_of_a_short_list():
    assert total([1, 2, 3]) == 6


def test_average_of_two_numbers():
    assert average([2, 4]) == 3


def test_highest_with_only_negatives():
    assert highest([-5, -2, -9]) == -2


def test_lowest_with_only_negatives():
    assert lowest([-5, -2, -9]) == -9


def test_highest_of_a_single_value():
    assert highest([7]) == 7

Defining them does nothing, exactly like any other function. Something has
to call them. Call one by hand:

In [ ]:
test_highest_with_only_negatives()
print("no AssertionError, so it passed")

## Testing that an error *does* happen

From the last notebook: `average([])` should raise a `ZeroDivisionError`,
and that is the behaviour we decided we want. So test it. The test passes
only if the error happens.

Written by hand, with the `try` / `except` from the last notebook:

In [ ]:
def test_average_of_an_empty_list_raises():
    try:
        average([])
    except ZeroDivisionError:
        return                                  # the error happened: pass
    raise AssertionError("average([]) did not raise ZeroDivisionError")


def test_highest_of_an_empty_list_raises():
    try:
        highest([])
    except IndexError:
        return
    raise AssertionError("highest([]) did not raise IndexError")


test_average_of_an_empty_list_raises()
test_highest_of_an_empty_list_raises()
print("both passed")

`raise` is new. It is how you cause an error on purpose. Here it means "if
we got this far, the error we expected never came, so fail".

That is six lines for one idea. pytest gives you a two-line version, which
is what the real test file uses:

```python
import pytest

def test_average_of_an_empty_list_raises():
    with pytest.raises(ZeroDivisionError):
        average([])
```

Read it as: "inside this block, I expect a `ZeroDivisionError`".

## A tiny test runner, to see what pytest does

Seven test functions. Calling each one by hand is tedious, and the first
failure would stop the cell. So: a loop that calls each one and counts.
This is, in miniature, what pytest does for you.

In [ ]:
TESTS = [
    test_total_of_a_short_list,
    test_average_of_two_numbers,
    test_highest_with_only_negatives,
    test_lowest_with_only_negatives,
    test_highest_of_a_single_value,
    test_average_of_an_empty_list_raises,
    test_highest_of_an_empty_list_raises,
]


def run_tests(tests):
    """Call every test, report each one, and return how many failed."""
    failed = 0
    for test in tests:
        try:
            test()
            print(f"PASSED  {test.__name__}")
        except AssertionError:
            print(f"FAILED  {test.__name__}")
            failed += 1
    print(f"{len(tests) - failed} passed, {failed} failed")
    return failed


assert run_tests(TESTS) == 0

A function is a value like any other, so it can go in a list, and
`test.__name__` is the name it was defined with.

## Now break something on purpose

Bring back the session 2 bug: start `highest` at `0` instead of at the
first value. Then run the same tests.

In [ ]:
def highest(numbers):
    """Return the largest value in a list of numbers. (Broken on purpose.)"""
    best = 0
    for number in numbers:
        if number > best:
            best = number
    return best


assert run_tests(TESTS) == 2

Two failures, and the second one is the more interesting. The negatives test
you expected. But `test_highest_of_an_empty_list_raises` failed too: with
`best = 0`, `highest([])` no longer crashes, it quietly answers `0`. The
test for the awkward case caught a second lie the first test could not see.

That is the whole case for testing the awkward cases on purpose. Put the
correct version back:

In [ ]:
def highest(numbers):
    """Return the largest value in a list of numbers."""
    best = numbers[0]
    for number in numbers:
        if number > best:
            best = number
    return best


assert run_tests(TESTS) == 0

## The real thing: a test file, and `python -m pytest`

In a project the tests live in their own file, next to the code they test:

```
sessions/session-12/demos/
    statistics_functions.py          the four functions, nothing else
    test_statistics_functions.py     the tests, which import them
```

Open both in VS Code now. The test file starts with

```python
import pytest

from statistics_functions import average, highest, lowest, total
```

and then has the same test functions as above, plus one that checks the
session 3 numbers end to end.

You run it from a terminal, in the repository root:

```
python -m pytest sessions/session-12/demos
```

`python -m pytest` rather than plain `pytest`, for the same reason as
`python -m pip`: it guarantees the pytest that runs is the one installed in
the Python you are using, which matters the moment you have a virtual
environment.

### If pytest is not installed

pytest is not part of Python; it is a package, like pandas. Install it into
your **project's** virtual environment (session 10), with the environment
active:

```
python -m pip install pytest
```

The next cell runs that terminal command from inside the notebook, so you
can see the output here. It checks first whether pytest is there, and tells
you what to do if not. Normally you would just type the command in a
terminal.

In [ ]:
def run_pytest(folder):
    """Run pytest on a folder and print what it says, if pytest is installed."""
    if importlib.util.find_spec("pytest") is None:
        print("pytest is not installed in this Python. In a terminal, with your")
        print("virtual environment active:  python -m pip install pytest")
        return None
    result = subprocess.run(
        [sys.executable, "-m", "pytest", str(folder), "-p", "no:cacheprovider"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    return result.returncode


code = run_pytest("sessions/session-12/demos")
assert code in (0, None)            # 0 means every test passed

Each dot is a passing test. `9 passed` at the bottom is the line you look
for.

(`-p no:cacheprovider` only stops pytest leaving a `.pytest_cache` folder
behind. You do not need it when you type the command yourself.)

### What a failure looks like

Same experiment as before, with real pytest: a copy of the module with the
`best = 0` bug, in `output/`, which git ignores.

In [ ]:
broken = Path("output/session-12/broken")
broken.mkdir(parents=True, exist_ok=True)
demos = Path("sessions/session-12/demos")

source = (demos / "statistics_functions.py").read_text(encoding="utf-8")
buggy = source.replace("best = numbers[0]", "best = 0")
assert buggy != source                     # make sure the bug actually went in
(broken / "statistics_functions.py").write_text(buggy, encoding="utf-8")
shutil.copy(demos / "test_statistics_functions.py", broken)

code = run_pytest(broken)
assert code in (1, None)            # 1 means some tests failed

# Tidy up, so a later `python -m pytest` never finds two modules with the
# same name and gets confused about which one it is testing.
shutil.rmtree(broken)

How to read that report, top to bottom:

* `..F....F.` is one character per test: `.` passed, `F` failed
* under **FAILURES**, each failed test is shown with the line that failed
  marked by `>`
* the `E` lines say why: `assert 0 == -2`, and helpfully, `where 0 =
  highest([-5, -2, -9])`. You did not have to print anything to find out
  what `highest` returned
* `DID NOT RAISE <class 'IndexError'>` is `pytest.raises` failing: the
  error you said would happen did not
* the last line is the count: `2 failed, 7 passed`

Compare that with a bare `assert` at the bottom of a file, which stops at
the first failure and only says `AssertionError`.

## What makes a good test

* **One idea per test**, named as a sentence. When
  `test_highest_with_only_negatives` fails, the name already tells you what
  broke.
* **The awkward cases**: empty, one item, negatives, zero, a missing value.
  The ordinary case nearly always works; the bugs live at the edges.
* **Small data you can check in your head.** `[-5, -2, -9]`, not the whole
  listening log. If you need a calculator to know the right answer, the
  test is too big.
* **Test what returns.** You can only test a function that hands back its
  answer. A function that prints instead cannot be checked by a test at
  all. That is the session 3 rule paying out one more time.

## In your project

Your project's `pipeline.py` has functions that calculate (the clean and
analyse steps) and an outer layer (`main()`) that loads and saves. Test the
calculating functions. They are where the decisions live.

1. Activate the project's virtual environment and install pytest:
   `python -m pip install pytest`.
2. Add `pytest` to `project/requirements.txt`, so the next person (or the
   next you, on a new laptop) gets it too.
3. Create `project/test_pipeline.py`, next to `pipeline.py`, so
   `from pipeline import ...` finds it.
4. Build tiny DataFrames by hand in the tests, with three or four rows whose
   right answer you know, rather than loading the real file:

```python
import pandas as pd

from pipeline import clean          # whatever your clean step is called


def test_clean_lowercases_genre():
    raw = pd.DataFrame({"genre": ["Jazz", " jazz", "JAZZ "]})
    assert list(clean(raw)["genre"]) == ["jazz", "jazz", "jazz"]
```

5. From the `project/` folder, run `python -m pytest`. With no folder
   named, pytest searches the current folder and everything under it.

`homework/test_pipeline_template.py` in this session's folder is a starter
for step 3.

## Next

`03-robust-exercise.ipynb`: make three small functions robust with
`try` / `except`, then write the tests that prove it.